# Inspect Development Pipeline Results

**Caveat:** these outputs validate the pipeline using development/Kaggle-style labels. They are not final thesis evaluation results. Final thesis results require the official ADReSS participant labels and splits.

This notebook loads the generated result CSVs and helps inspect baseline performance, optimized performance, shuffled-label controls, SHAP outputs, and stability metrics.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUTS = PROJECT_ROOT / "outputs"
FIGURES = OUTPUTS / "figures"

files = {
    "features": OUTPUTS / "features.csv",
    "baseline": OUTPUTS / "baseline_results.csv",
    "optimized": OUTPUTS / "optimized_results.csv",
    "shuffled": OUTPUTS / "shuffled_label_results.csv",
    "shap_baseline": OUTPUTS / "shap_values_baseline.csv",
    "shap_optimized": OUTPUTS / "shap_values_optimized.csv",
    "stability": OUTPUTS / "stability_results.csv",
}

for name, path in files.items():
    print(f"{name:15s} exists={path.exists()} path={path}")

print("\nFigures:")
for path in sorted(FIGURES.glob("*")):
    print(path.name)

## Load Tables

In [ ]:
features = pd.read_csv(files["features"])
baseline = pd.read_csv(files["baseline"])
optimized = pd.read_csv(files["optimized"])
shuffled = pd.read_csv(files["shuffled"])
stability = pd.read_csv(files["stability"])

print("features shape:", features.shape)
display(features.head())

print("baseline shape:", baseline.shape)
display(baseline)

print("optimized shape:", optimized.shape)
display(optimized)

print("shuffled-label shape:", shuffled.shape)
display(shuffled)

print("stability shape:", stability.shape)
display(stability)

## Performance Summary

Use AUROC as the main quick check, but also inspect F1, precision, and recall. Zero precision/recall usually means the model predicted no dementia cases in one or more folds.

In [ ]:
metric_cols = [
    "model",
    "variant",
    "roc_auc_mean",
    "roc_auc_std",
    "accuracy_mean",
    "f1_mean",
    "precision_mean",
    "recall_mean",
]

performance = pd.concat(
    [baseline[metric_cols], optimized[metric_cols]],
    ignore_index=True,
).sort_values(["roc_auc_mean", "f1_mean"], ascending=False)

display(performance)

## Baseline vs Optimized Delta

In [ ]:
compare_cols = ["roc_auc_mean", "accuracy_mean", "f1_mean", "precision_mean", "recall_mean"]

base_i = baseline.set_index("model")[compare_cols].add_prefix("baseline_")
opt_i = optimized.set_index("model")[compare_cols].add_prefix("optimized_")
delta = base_i.join(opt_i, how="outer")

for col in compare_cols:
    delta[f"delta_{col}"] = delta[f"optimized_{col}"] - delta[f"baseline_{col}"]

display(delta.sort_values("delta_roc_auc_mean", ascending=False))

## Shuffled-Label Control

The shuffled-label control should perform near chance. If real-label results are not clearly better than shuffled labels, treat the development result as weak evidence.

In [ ]:
shuffled_summary = shuffled[metric_cols].sort_values("roc_auc_mean", ascending=False)
display(shuffled_summary)

real_vs_shuffled = baseline.set_index("model")[["roc_auc_mean", "f1_mean"]].join(
    shuffled.set_index("model")[["roc_auc_mean", "f1_mean"]],
    lsuffix="_real_baseline",
    rsuffix="_shuffled",
)
real_vs_shuffled["delta_roc_auc_real_minus_shuffled"] = (
    real_vs_shuffled["roc_auc_mean_real_baseline"] - real_vs_shuffled["roc_auc_mean_shuffled"]
)
real_vs_shuffled["delta_f1_real_minus_shuffled"] = (
    real_vs_shuffled["f1_mean_real_baseline"] - real_vs_shuffled["f1_mean_shuffled"]
)
display(real_vs_shuffled.sort_values("delta_roc_auc_real_minus_shuffled", ascending=False))

## Stability Metrics

In [ ]:
stability_pivot = stability.pivot_table(
    index=["model", "variant"],
    columns=["scope", "metric"],
    values="value",
    aggfunc="first",
)
display(stability_pivot)

## SHAP Output Checks

In [ ]:
shap_baseline = pd.read_csv(files["shap_baseline"])
shap_optimized = pd.read_csv(files["shap_optimized"])

print("shap baseline shape:", shap_baseline.shape)
print("shap optimized shape:", shap_optimized.shape)

display(shap_baseline.head())
display(shap_optimized.head())

for label, df in [("baseline", shap_baseline), ("optimized", shap_optimized)]:
    print(f"\nTop SHAP rows for {label}:")
    value_col = "mean_abs_shap" if "mean_abs_shap" in df.columns else df.select_dtypes("number").columns[-1]
    display(df.sort_values(value_col, ascending=False).head(15))

## Notes To Carry Forward

- These are development/Kaggle-style label results, not final ADReSS thesis results.
- Check whether optimized AUROC improves over baseline and whether F1/recall remain near zero.
- Check whether real-label baselines outperform shuffled-label controls.
- Check whether top SHAP features are stable across baseline/optimized variants.